# 04 – Test kết nối mạng Supabase & Database

Kiểm tra cấu hình biến môi trường, ping REST API của Supabase, kiểm tra cổng PostgreSQL local và ghi chú về `pgvector`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent

load_dotenv(root / ".env")

for name in ("SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY", "DATABASE_URL"):
    status = "có" if os.getenv(name) else "thiếu"
    print(f"{name}: {status}")

In [ ]:
import time

import httpx

supabase_url = os.environ["SUPABASE_URL"].rstrip("/")
anon_key = os.environ["SUPABASE_ANON_KEY"]
service_key = os.environ["SUPABASE_SERVICE_ROLE_KEY"]

for label, key in (("anon_key", anon_key), ("service_role_key", service_key)):
    headers = {"apikey": key, "Authorization": f"Bearer {key}"}
    t0 = time.perf_counter()
    resp = httpx.get(f"{supabase_url}/rest/v1/", headers=headers, timeout=10.0)
    latency_ms = (time.perf_counter() - t0) * 1000
    print(f"Supabase REST ({label}): HTTP {resp.status_code} ({latency_ms:.1f} ms)")

In [ ]:
t0 = time.perf_counter()
auth_resp = httpx.get(
    f"{supabase_url}/auth/v1/health",
    headers={"apikey": anon_key},
    timeout=10.0,
)
latency_ms = (time.perf_counter() - t0) * 1000
print(f"Supabase Auth health: HTTP {auth_resp.status_code} ({latency_ms:.1f} ms)")

In [ ]:
import socket
from urllib.parse import urlparse

db_url = os.getenv("DATABASE_URL", "")
parsed = urlparse(db_url)
host = parsed.hostname or "localhost"
port = parsed.port or 5432

t0 = time.perf_counter()
try:
    with socket.create_connection((host, port), timeout=3.0):
        latency_ms = (time.perf_counter() - t0) * 1000
        print(f"Postgres TCP ({host}:{port}): kết nối OK ({latency_ms:.1f} ms)")
except OSError as exc:
    print(f"Postgres TCP ({host}:{port}): không kết nối được ({exc})")

## Ghi chú nhanh: Supabase vs Postgres local cho VectorDB (`pgvector`)

- **Supabase có `pgvector`**: bật trong Dashboard → *Database → Extensions* (`vector`) hoặc `CREATE EXTENSION IF NOT EXISTS vector;`.
- **Giới hạn 2000 chiều index HNSW**: `pgvector` chỉ tạo được index HNSW/IVFFlat cho kiểu `vector` tối đa 2000 chiều. Với vector 2048 chiều, khi dữ liệu lớn cần chuyển sang `halfvec(2048)` hoặc giảm số chiều xuống `<= 2000` (áp dụng cho cả local lẫn Supabase).
- **Dung lượng & độ trễ**: Supabase Free giới hạn 500 MB database và ngủ sau 7 ngày không hoạt động; Postgres local (Docker) không giới hạn 500 MB và có độ trễ thấp hơn, giữ dữ liệu nội bộ.